In [1]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta
from sqlalchemy import text
import numpy as np
import oracledb
import sys

#CONEXIONES DESTINO

DB_USER=config("USER2_BDI_DW")
DB_PASSWORD=config("PASS2_BDI_DW")
DB_NAME="INDICADORES_ESSALUD"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_DW")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb

# Configuración de la conexión a Oracle
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname = config("HOST4_BDI_POSTGRES")
service_name = "WNET"
port = 1527

engine0 = create_engine(f'oracle://{un}:{pw}@', connect_args={
    "host": hostname,
    "port": port,
    "service_name": service_name
})

connection0 = engine0.connect()

# Cargar datos de referencia desde PostgreSQL
tipdoc = pd.read_sql_query("SELECT id_tipdoc, cod_tdo FROM dim_tipdoc", con=engine2)
cas = pd.read_sql_query(f"SELECT id_cas,cod_cas, cod_red FROM dim_cas where cod_cas is not null", con=connection2)
gruocu = pd.read_sql_query("SELECT id_gruocu, cod_gru FROM dim_gruocu", con=engine2)
condtrab = pd.read_sql_query("SELECT id_condtrab, cod_condtrab FROM dim_condtrab", con=engine2)
estreg = pd.read_sql_query("SELECT id_estreg, cod_est FROM dim_estreg", con=engine2)

# Función para eliminar espacios en blanco
def strip_columns(df):
    return df.apply(lambda x: x.str.strip() if x.dtype == "object" and x.apply(type).eq(str).all() else x)

# Aplica la función a cada dataframe
tipdoc = strip_columns(tipdoc)
cas = strip_columns(cas)
gruocu = strip_columns(gruocu)
condtrab = strip_columns(condtrab)
estreg = strip_columns(estreg)

# Obtener y ajustar la fecha de inicio
fec_ini = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_proceso=2", con=connection2)
fec_ini = fec_ini.iloc[0, 0] - timedelta(days=2)

# Consulta para obtener los datos a partir de fec_ini
query = f"""
    SELECT
        prs.TIPDOCIDENPERCOD,
        prs.PERASISDOCIDENNUM,
        prs.PERASISPROAPEPAT,
        prs.PERASISPROAPEMAT,
        prs.PERASISPROPRINOM,
        prs.PERASISPROSEGNOM,
        prs.GRUPOCUPCOD,
        prs.CONDTRABCOD,
        prs.PERASISPROCOLCOD,
        pas.CENASICOD,
        prs.PERASISESTREGCOD, 
        prs.PERASISPROFECCRE,
        prs.PERASISPROFECMOD
    FROM CMPRS10 prs
    LEFT OUTER JOIN CMPAS10 pas
        ON prs.PERASISDOCIDENNUM = pas.PERASISDOCIDENNUM
    WHERE prs.PERASISPROFECCRE >= TO_DATE('{fec_ini.strftime('%Y-%m-%d')}', 'YYYY-MM-DD') 
        OR prs.PERASISPROFECMOD >= TO_DATE('{fec_ini.strftime('%Y-%m-%d')}', 'YYYY-MM-DD')
"""

# Cargar todos los datos en un DataFrame
chunk = pd.read_sql_query(query, con=engine0)

# Renombrar columnas
chunk = chunk.rename(columns={
    'tipdocidenpercod': 'cod_tdo',
    'perasisdocidennum': 'num_doc',
    'perasisproapepat': 'ape_pat',
    'perasisproapemat': 'ape_mat',
    'perasisproprinom': 'pri_nom',
    'perasisprosegnom': 'seg_nom',
    'grupocupcod': 'cod_gru',
    'condtrabcod': 'cod_condtrab',
    'perasisprocolcod': 'cod_col',
    'cenasicod': 'cod_cas',
    'perasisestregcod': 'cod_est', 
    'perasisprofeccre': 'fec_cre',
    'perasisprofecmod': 'fec_mod'
})

# Eliminar espacios en blanco
chunk = strip_columns(chunk)

# Realizar merges con los DataFrames de referencia
chunk = pd.merge(chunk, tipdoc, on='cod_tdo', how='left').drop('cod_tdo', axis=1)
chunk = pd.merge(chunk, cas, on='cod_cas', how='left').drop('cod_cas', axis=1)
chunk = pd.merge(chunk, gruocu, on='cod_gru', how='left').drop('cod_gru', axis=1)
chunk = pd.merge(chunk, condtrab, on='cod_condtrab', how='left').drop('cod_condtrab', axis=1)
chunk = pd.merge(chunk, estreg, on='cod_est', how='left').drop('cod_est', axis=1)

# Eliminar duplicados
chunk.drop_duplicates(subset=['num_doc'], inplace=True)

# Convertir columnas de fecha a datetime
date_columns = ['fec_cre', 'fec_mod']
for col in date_columns:
    chunk[col] = pd.to_datetime(chunk[col], errors='coerce')
    chunk[col] = chunk[col].apply(lambda x: None if pd.isna(x) else x)

# Convertir el resto de NaN a None
chunk = chunk.applymap(lambda x: None if pd.isna(x) else x)

# Verificar e insertar o actualizar registros
for _, row in chunk.iterrows():
    row_dict = row.to_dict()
    
    # Reemplazar valores 'NaT' en fechas con None
    for key in date_columns:
        if row_dict[key] == 'NaT' or pd.isna(row_dict[key]):
            row_dict[key] = None

    # Verificar si el registro ya existe
    check_query = text("""
    SELECT 1 FROM dim_personal WHERE num_doc = :num_doc
    """)
    result = engine2.execute(check_query, num_doc=row_dict['num_doc']).fetchone()

    if result:
        # Si existe, actualizar el registro
        update_query = text("""
        UPDATE dim_personal
        SET id_tipdoc=:id_tipdoc, pri_nom=:pri_nom, seg_nom=:seg_nom, ape_pat=:ape_pat, ape_mat=:ape_mat, 
            id_cas=:id_cas, fec_cre=:fec_cre, fec_mod=:fec_mod, id_gruocu=:id_gruocu, id_condtrab=:id_condtrab, 
            cod_col=:cod_col, id_estreg=:id_estreg
        WHERE num_doc = :num_doc
        """)
        engine2.execute(update_query, **row_dict)
    else:
        # Si no existe, insertar un nuevo registro
        insert_query = text("""
        INSERT INTO dim_personal (id_tipdoc, num_doc, pri_nom, seg_nom, ape_pat, ape_mat, id_cas, 
                                fec_cre, fec_mod, id_gruocu, id_condtrab, cod_col, id_estreg)
        VALUES (:id_tipdoc, :num_doc, :pri_nom, :seg_nom, :ape_pat, :ape_mat, :id_cas, :fec_cre, :fec_mod, 
                :id_gruocu, :id_condtrab, :cod_col, :id_estreg)
        """)
        engine2.execute(insert_query, **row_dict)

In [2]:
now = datetime.now().strftime('%Y-%m-%d')
query1=f"UPDATE etl_act SET fec_ini ='{now}' WHERE id_proceso=2"
query2=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_proceso=2"
c1= text(query1)
c2= text(query2)
connection2.execute(c1)
connection2.execute(c2)

In [3]:
# Cerrar conexiones
connection2.close()
connection0.close()
engine2.dispose()
engine0.dispose()